# Week 2 – Markdown, NumPy, plotting, curve fitting and MNIST

**What the lab wanted:** use Markdown, import libraries like Matplotlib to plot graphs, load a dataset (MNIST) and output results. The polynomial curve fitting notebook (`PCF-JN`) is the worked example.

By the end you should understand: NumPy arrays and shapes, how to plot, what "fitting a model" means, what **overfitting** is, and what the MNIST data actually looks like.

## 1. Markdown in 60 seconds
Double-click this cell to see how it's written.

# Big heading  (one `#`)
## Smaller heading  (two `##`)
**bold** with `**double stars**`, *italics* with `*single stars*`, `code` with backticks.

* bullet points start with `*` or `-`
* maths goes between dollar signs: $y = 2x + 5$, or on its own line with double dollars:

$$L = -\ln P(Y = y_i \mid X = x_i)$$

To turn a cell into Markdown in VS Code: click the cell, press **Esc** then **M** (press **Y** to turn it back into code).

## 2. NumPy – the foundation of everything
NumPy gives you **arrays**: grids of numbers that you can do maths on all at once (no for loops needed). PyTorch tensors in Week 4 behave almost identically, so time spent here pays off.

In [ ]:
import numpy as np

a = np.array([1.0, 2.0, 3.0])
print(a * 10)          # every element times 10  -> [10. 20. 30.]
print(a + a)           # element-wise addition
print(a ** 2)          # element-wise square
print(np.sum(a), np.max(a), np.mean(a))

### Shape: the most important property of an array
`.shape` tells you the size in each direction. **Most errors in AI code are shape errors**, so get in the habit of printing shapes.

In [ ]:
v = np.array([1, 2, 3])            # 1-D: shape (3,)
M = np.array([[1, 2, 3],
              [4, 5, 6]])          # 2-D: 2 rows, 3 columns -> shape (2, 3)
print("v.shape =", v.shape)
print("M.shape =", M.shape)
print("M[1, 2] =", M[1, 2])        # row 1, column 2 (counting from 0) -> 6
print("M[:, 0] =", M[:, 0])        # ':' means 'all' -> first column [1 4]

### Handy ways to make arrays
* `np.linspace(start, stop, n)` – `n` evenly spaced numbers from start to stop (inclusive).
* `np.random.rand(...)` – random numbers between 0 and 1.
* `np.random.normal(mean, std, n)` – random "noise" from a bell curve.

In [ ]:
x = np.linspace(-10, 10, 5)
print(x)                                  # [-10.  -5.   0.   5.  10.]
print(np.random.rand(2, 3))               # 2x3 grid of random numbers in [0, 1)
print(np.random.normal(0, 0.1, 4))        # 4 small random wiggles around 0

## 3. Plotting with Matplotlib
The standard import is `import matplotlib.pyplot as plt`. (The lab notebook called it `mp` – same thing, just a different nickname.)

`plt.plot(x, y)` draws a line through the points. Add `'o'` to draw dots instead of a line.

In [ ]:
import matplotlib.pyplot as plt

x = np.linspace(0, 2 * np.pi, 100)
plt.plot(x, np.sin(x), label="sin(x)")
plt.plot(x, np.cos(x), label="cos(x)")
plt.plot([1, 2, 3], [0.5, -0.5, 0], 'o', label="some dots")
plt.xlabel("x"); plt.ylabel("y"); plt.title("My first plot")
plt.legend(); plt.grid()
plt.show()

## 4. Polynomial curve fitting (the PCF notebook, explained)
This is your first real **machine learning** example, even though it doesn't look like it.

**The setup:** we have some data points $(x, y)$. We want a formula (a "model") that predicts $y$ from $x$.
We choose the model to be a polynomial, e.g. degree 2: $y = a x^2 + b x + c$.
**"Fitting"** means finding the numbers $a, b, c$ that make the curve pass as close as possible to the points. Those numbers are the model's **parameters** (in a neural network they're called **weights**).

* `np.polyfit(x, y, degree)` – does the fitting, returns the coefficients (highest power first).
* `np.poly1d(coeff)` – turns those coefficients into a function you can call, e.g. `yn(3.0)`.

In [ ]:
x = np.linspace(-10, 10, 5)        # only 5 data points
y = 2 * x + 5                      # the 'true' relationship is a straight line

coeff = np.polyfit(x, y, 2)        # fit a degree-2 polynomial
print("coefficients [a, b, c]:", np.round(coeff, 4))   # ~[0, 2, 5]: it found the line!

yn = np.poly1d(coeff)              # yn is now a function
xn = np.linspace(-20, 20, 100)     # lots of x values to draw a smooth curve
plt.plot(xn, yn(xn), label="fitted curve")
plt.plot(x, y, 'o', label="data")
plt.legend(); plt.grid(); plt.show()

Even though we allowed a curve (degree 2), polyfit found $a \approx 0$ because the data is a straight line.

### When the model is too simple: underfitting
Now the data comes from $y = 2x^3 + 5$ (a cubic) but we only allow degree 2. The curve **can't** bend the right way, so it misses the points. This is **underfitting**.

In [ ]:
y = 2 * x**3 + 5
for degree in [2, 3]:
    yn = np.poly1d(np.polyfit(x, y, degree))
    plt.plot(xn, yn(xn), label=f"degree {degree}")
plt.plot(x, y, 'o', label="data")
plt.ylim(-3000, 3000); plt.legend(); plt.grid(); plt.show()
# degree 2 misses badly, degree 3 goes through every point

### Noisy data and the error measure
Real data is noisy. Below, the true curve is $\sin(2\pi x)$ but every point has random noise added.

To say how good a fit is we use the **sum of squared errors**: for every point, take (prediction − actual), square it, add them all up:
$$\text{error} = \sum_i \big(\hat y_i - y_i\big)^2$$
Smaller is better. This is your first **loss function** – the idea comes back every week.

> ⚠️ **Bug in the lab notebook:** it uses `math.pi` without `import math`, makes `x` with 5 points but adds 50 noise values, and computes the error with `yn(xn) - y` where `xn` and `y` are different lengths. That's why it may crash for you. The version below is fixed.

In [ ]:
np.random.seed(0)                                   # makes the 'random' noise the same every run
x = np.linspace(-1, 1, 50)                          # 50 points this time
y = np.sin(2 * np.pi * x) + np.random.normal(0, 0.1, 50)
x_dense = np.linspace(-1, 1, 300)                   # for drawing smooth curves

def sse(model, x, y):
    # sum of squared errors of a fitted polynomial on points (x, y)
    return np.sum((model(x) - y) ** 2)

for degree in [3, 8]:
    yn = np.poly1d(np.polyfit(x, y, degree))
    plt.plot(x_dense, yn(x_dense), label=f"degree {degree}  (error {sse(yn, x, y):.2f})")
plt.plot(x_dense, np.sin(2 * np.pi * x_dense), 'k--', label="true curve")
plt.plot(x, y, 'o', ms=3, label="noisy data")
plt.legend(); plt.grid(); plt.show()

Degree 3 is too stiff (big error). Degree 8 follows the sine wave nicely (small error).

### When the model is too flexible: overfitting
Now the important bit. Take only **10** noisy points and fit a degree-9 polynomial. With 10 coefficients and 10 points it can pass through **every single point exactly** – error ≈ 0.

But is it a good model? To find out, we generate **new** data from the same process (a **test set**) and check the error there.

In [ ]:
np.random.seed(1)
x = np.linspace(-1, 1, 10)
y_train = np.sin(2 * np.pi * x) + np.random.normal(0, 0.2, 10)
y_test  = np.sin(2 * np.pi * x) + np.random.normal(0, 0.2, 10)   # fresh noise = new data

yn = np.poly1d(np.polyfit(x, y_train, 9))
print(f"error on TRAINING data: {sse(yn, x, y_train):.6f}")       # basically zero
print(f"error on TEST data:     {sse(yn, x, y_test):.4f}")         # much bigger

plt.plot(x_dense, yn(x_dense), label="degree 9 fit")
plt.plot(x_dense, np.sin(2 * np.pi * x_dense), 'k--', label="true curve")
plt.plot(x, y_train, 'o', label="training points")
plt.plot(x, y_test, 'x', label="test points")
plt.ylim(-3, 3); plt.legend(); plt.grid(); plt.show()

**This is the most important idea in the whole module:**
> A model that's perfect on the data it was trained on can still be bad on new data. It has memorised the noise instead of learning the pattern. That's **overfitting**.

That's why in Week 4 PyTorch the MNIST data is split into a **training set** (used to fit the weights) and a **test set** (never trained on, only used to measure accuracy). Always judge a model on data it hasn't seen.

### Try it
Change the degree in the cell above to 1, 3, 5 and 9. Which gives the lowest **test** error? (Usually somewhere in the middle – not too simple, not too flexible.)

## 5. Loading the MNIST dataset
MNIST is 70,000 small greyscale images of handwritten digits 0–9 (60,000 for training, 10,000 for testing). Each image is 28 × 28 pixels. It's the "hello world" of image classification, and it's what you classify in Week 4.

`torchvision` (part of PyTorch) downloads it for you. The first run downloads ~50 MB into a folder called `data` next to this notebook.

In [ ]:
from torchvision import datasets

train_data = datasets.MNIST(root="./data", train=True,  download=True)
test_data  = datasets.MNIST(root="./data", train=False, download=True)
print("training images:", len(train_data))
print("test images:    ", len(test_data))

Each item in the dataset is a pair: **(image, label)**. The label is the correct answer (the digit).

In [ ]:
image, label = train_data[0]          # the first example
print("label:", label)
print("image type:", type(image))     # a PIL image (a normal picture object)

pixels = np.array(image)              # convert to a numpy array to look inside
print("shape:", pixels.shape)         # (28, 28)
print("min/max pixel:", pixels.min(), pixels.max())   # 0 = black, 255 = white

An image is **just a grid of numbers**. Here's the top-left corner of the digit as raw numbers, then drawn as a picture with `plt.imshow`:

In [ ]:
print(pixels[5:15, 5:15])            # a 10x10 patch of the image
plt.imshow(pixels, cmap="gray")
plt.title(f"label = {label}")
plt.show()

### Output results: a grid of examples
`plt.subplots(rows, cols)` makes a grid of mini-plots. `fig.savefig(...)` saves the figure as a file (that's the "output results" part of the lab).

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, ax in enumerate(axes.flat):         # enumerate gives (index, item)
    img, lab = train_data[i]
    ax.imshow(img, cmap="gray")
    ax.set_title(str(lab))
    ax.axis("off")
plt.tight_layout()
fig.savefig("mnist_examples.png")          # saved next to this notebook
plt.show()

In [ ]:
# How many of each digit are in the training set?
labels = train_data.targets.numpy()        # all 60,000 labels as a numpy array
counts = np.bincount(labels)               # counts how many 0s, 1s, 2s, ...
for digit, n in enumerate(counts):
    print(f"digit {digit}: {n} images")
plt.bar(range(10), counts); plt.xlabel("digit"); plt.ylabel("count"); plt.show()

## Week 2 checklist
- I can make arrays, check `.shape`, and do maths on whole arrays at once.
- I can plot lines and dots with labels and a legend, and show images with `imshow`.
- I understand **fitting** = finding the parameters that make the error small.
- I can explain **underfitting**, **overfitting**, and why we keep a separate **test set**.
- I know an MNIST example is (28×28 grid of pixel values, digit label).

Next: **Week 3** – how a model turns an image into scores, and how we measure how wrong it is.